# Taller III. Analítica académica con SQL sobre university.db

**Especialización en Inteligencia Artificial, UPTC**

**Herramientas:** Python, con el módulo `sqlite3` para ejecutar las consultas sobre el motor SQLite y pandas para mostrar los resultados.

## Contexto y objetivo

La universidad quiere responder preguntas estratégicas sobre rendimiento estudiantil, ocupación de cursos, distribución docente, matrículas y uso de la infraestructura física. La información está en una base de datos relacional normalizada (`university.db`). En este taller se resuelve una batería progresiva de consultas SQL (exploración, manipulación de datos, cruces, auditoría de registros huérfanos y agregaciones) y dos retos integradores, uno de ellos sobre la auditoría de una consulta generada por un agente de IA (Text-to-SQL).

Cada consulta se presenta con el **código SQL**, la **evidencia de ejecución** (tabla de resultados del motor SQLite) y una **interpretación analítica** desde la gestión académica.

## Conexión a la base de datos

Se usa el motor SQLite a través del módulo `sqlite3` de Python. La función `consulta` ejecuta sentencias `SELECT` y muestra el resultado como tabla junto con el número de filas devueltas; la función `ejecutar` aplica sentencias `INSERT`, `UPDATE` y `DELETE`, confirma el cambio (`commit`) y reporta las filas afectadas.

In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

ruta_db = 'university.db'
if EN_COLAB:
    drive.mount('/content/drive')
    ruta_db = '/content/drive/MyDrive/university.db'

conn = sqlite3.connect(ruta_db)


def consulta(sql):
    """Ejecuta una consulta SELECT y muestra el resultado como tabla."""
    resultado = pd.read_sql_query(sql, conn).convert_dtypes()
    display(resultado.style.hide(axis='index').format(precision=1, decimal=',', na_rep='NULL'))
    print(f'Filas devueltas: {len(resultado)}')


def ejecutar(sql):
    """Ejecuta una sentencia INSERT, UPDATE o DELETE y confirma el cambio."""
    cursor = conn.execute(sql)
    conn.commit()
    print(f'Filas afectadas: {cursor.rowcount}')
    if sql.strip().upper().startswith('INSERT'):
        print(f'ID asignado al nuevo registro: {cursor.lastrowid}')


print(f'Conectado a {ruta_db} con SQLite {sqlite3.sqlite_version}')

Conectado a university.db con SQLite 3.49.1


## Modelo relacional y diccionario de datos

Antes de consultar, se revisa la estructura real de la base: tablas, número de registros, llaves primarias y llaves foráneas, leídas del catálogo de SQLite.

In [2]:
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name", conn)['name']

diccionario = []
for tabla in tablas:
    columnas = pd.read_sql_query(f"SELECT name, pk FROM pragma_table_info('{tabla}')", conn)
    registros = conn.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]
    diccionario.append({'TABLA': tabla, 'REGISTROS': registros,
                        'LLAVE_PRIMARIA': ', '.join(columnas.loc[columnas['pk'] > 0, 'name']),
                        'COLUMNAS': ', '.join(columnas['name'])})

diccionario = pd.DataFrame(diccionario)
display(diccionario.style.hide(axis='index'))
print(f"Tablas: {len(diccionario)} | Registros en total: {diccionario['REGISTROS'].sum()}")

TABLA,REGISTROS,LLAVE_PRIMARIA,COLUMNAS
CALENDARIO_CURSOS,72,ID_CALENDARIO,"ID_CALENDARIO, DIA, HORA_INICIO, HORA_FIN, CUPO, ID_CURSO, ID_SALON, ID_PROFESOR"
CARRERAS,4,ID_CARRERA,"ID_CARRERA, NOMBRE, ID_FACULTAD"
CARRERAS_ESTUDIANTES,10,"ID_CARRERA, ID_USUARIO","ID_CARRERA, ID_USUARIO"
CURSOS,71,ID_CURSO,"ID_CURSO, NOMBRE, ID_FACULTAD"
CURSOS_CARRERAS,101,"ID_CURSO, ID_CARRERA","ID_CURSO, ID_CARRERA, SEMESTRE"
CURSOS_ESTUDIANTES,151,"ID_CALENDARIO, ID_USUARIO","ID_CALENDARIO, ID_USUARIO"
EDIFICIOS,5,ID_EDIFICIO,"ID_EDIFICIO, NOMBRE"
FACULTADES,2,ID_FACULTAD,"ID_FACULTAD, NOMBRE"
ROLES,2,ID_ROL,"ID_ROL, NOMBRE"
SALONES,42,ID_SALON,"ID_SALON, NOMBRE, ID_EDIFICIO"


Tablas: 11 | Registros en total: 480


**Análisis.** La base tiene 11 tablas y 480 registros, tal como indica el taller. Las tablas de catálogo son pequeñas (2 facultades, 2 roles, 4 carreras y 5 edificios), mientras que las tablas de relación concentran la mayor parte de los datos: CURSOS_ESTUDIANTES con 151 inscripciones y CURSOS_CARRERAS con 101 asignaciones de la malla. Las tres tablas de relación usan llaves primarias compuestas, lo que impide, por ejemplo, inscribir dos veces al mismo estudiante en la misma sesión.

In [3]:
consulta("""
SELECT
    m.name    AS TABLA,
    f."from"  AS COLUMNA,
    f."table" AS TABLA_REFERENCIADA,
    f."to"    AS COLUMNA_REFERENCIADA
FROM sqlite_master m
INNER JOIN pragma_foreign_key_list(m.name) f
WHERE m.type = 'table'
ORDER BY m.name, f."from";
""")

TABLA,COLUMNA,TABLA_REFERENCIADA,COLUMNA_REFERENCIADA
CALENDARIO_CURSOS,ID_CURSO,CURSOS,ID_CURSO
CALENDARIO_CURSOS,ID_PROFESOR,USUARIOS,ID_USUARIO
CALENDARIO_CURSOS,ID_SALON,SALONES,ID_SALON
CARRERAS,ID_FACULTAD,FACULTADES,ID_FACULTAD
CARRERAS_ESTUDIANTES,ID_CARRERA,CARRERAS,ID_CARRERA
CARRERAS_ESTUDIANTES,ID_USUARIO,USUARIOS,ID_USUARIO
CURSOS,ID_FACULTAD,FACULTADES,ID_FACULTAD
CURSOS_CARRERAS,ID_CARRERA,CARRERAS,ID_CARRERA
CURSOS_CARRERAS,ID_CURSO,CURSOS,ID_CURSO
CURSOS_ESTUDIANTES,ID_CALENDARIO,CALENDARIO_CURSOS,ID_CALENDARIO


Filas devueltas: 13


**Análisis.** Hay 13 llaves foráneas que conectan todo el modelo. USUARIOS cumple dos papeles: es el estudiante en CURSOS_ESTUDIANTES y CARRERAS_ESTUDIANTES, y el profesor en CALENDARIO_CURSOS (columna ID_PROFESOR). Las llaves foráneas están declaradas, pero SQLite no las hace cumplir por defecto (`PRAGMA foreign_keys` está desactivado), así que la integridad referencial depende de cómo se cargaron los datos; por eso el Bloque D audita registros huérfanos.

## Parte I. Batería progresiva de consultas SQL

## Bloque A. Exploración, ordenamiento y filtros básicos

### A1. Mostrar todos los registros de la tabla USUARIOS

In [4]:
consulta("""
SELECT *
FROM USUARIOS;
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
1,Ben,Stone,0,2
2,Grace,Stone,0,2
3,Mikaela,Stone,0,2
4,Jared,Vásquez,0,2
5,Zeke,Landon,0,2
6,Saanvi,Bahl,0,2
7,Ulrich,Nielsen,0,2
8,Charlotte,Doppler,0,2
9,Claudia,Tiedemann,0,2
10,Robert,Vance,0,2


Filas devueltas: 20


**Análisis.** La tabla tiene 20 usuarios: los 10 primeros son profesores (`ID_ROL = 2`) y aparecen con semestre 0, y los 10 restantes son estudiantes (`ID_ROL = 1`) entre los semestres 1 y 6. El semestre 0 de los profesores es un valor de relleno que hay que excluir en cualquier cálculo sobre semestres (ver E3).

### A2. Mostrar únicamente el NOMBRE y APELLIDO de todos los usuarios

In [5]:
consulta("""
SELECT NOMBRE, APELLIDO
FROM USUARIOS;
""")

NOMBRE,APELLIDO
Ben,Stone
Grace,Stone
Mikaela,Stone
Jared,Vásquez
Zeke,Landon
Saanvi,Bahl
Ulrich,Nielsen
Charlotte,Doppler
Claudia,Tiedemann
Robert,Vance


Filas devueltas: 20


**Análisis.** Seleccionar solo las columnas necesarias reduce la consulta a la información útil. Varios apellidos se repiten (Stone 5 veces, Nielsen 4, Doppler 3 y Tiedemann 2), así que el apellido no sirve para identificar a una persona; para eso está `ID_USUARIO`.

### A3. Mostrar todas las carreras ordenadas alfabéticamente de forma ascendente

In [6]:
consulta("""
SELECT ID_CARRERA, NOMBRE, ID_FACULTAD
FROM CARRERAS
ORDER BY NOMBRE ASC;
""")

ID_CARRERA,NOMBRE,ID_FACULTAD
4,Filosofía,2
2,Ingeniería Mecánica,1
1,Ingeniería de Sistemas,1
3,Matemáticas Aplicadas,1


Filas devueltas: 4


**Análisis.** Las 4 carreras quedan en orden ascendente, pero Ingeniería Mecánica aparece antes que Ingeniería de Sistemas. La razón es que SQLite ordena por defecto con la intercalación BINARY, que compara los códigos de los caracteres: la M mayúscula va antes que la d minúscula. Para un orden alfabético que no distinga mayúsculas se puede usar `ORDER BY NOMBRE COLLATE NOCASE`.

### A4. Mostrar los cursos ordenados alfabéticamente de forma descendente

In [7]:
consulta("""
SELECT ID_CURSO, NOMBRE, ID_FACULTAD
FROM CURSOS
ORDER BY NOMBRE DESC;
""")

ID_CURSO,NOMBRE,ID_FACULTAD
67,Ética,2
9,Árboles y Grafos,1
49,Álgebra Moderna,1
45,Álgebra Lineal,1
4,Técnicas y Prácticas de Programación,1
38,Transferencia de Calor,1
70,Trabajo de Grado,NULL
56,Textos Filosóficos,2
33,Termodinámica,1
50,Teoría de Probabilidades,1


Filas devueltas: 71


**Análisis.** Se listan los 71 cursos. Por la misma intercalación BINARY, los nombres que empiezan con letra tildada (Ética, Árboles y Grafos, Álgebra Moderna y Álgebra Lineal) quedan al principio del orden descendente, porque sus códigos son mayores que los de cualquier letra sin tilde. La consulta también deja ver problemas de calidad del catálogo: dos cursos sin facultad (Trabajo de Grado y Práctica Estudiantil, con `ID_FACULTAD` nulo) y errores de digitación como Desarrollo de Videouegos y Estadítica Aplicada.

### A5. Mostrar los estudiantes que están cursando el semestre 2

In [8]:
consulta("""
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE
FROM USUARIOS
WHERE ID_ROL = 1
  AND SEMESTRE = 2;
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
20,Mads,Nielsen,2


Filas devueltas: 1


**Análisis.** Solo Mads Nielsen está en segundo semestre. Se filtra también por `ID_ROL = 1` para asegurar que el resultado sean estudiantes y no otro tipo de usuario.

### A6. Mostrar los usuarios cuyo apellido sea 'Gomez'

In [9]:
consulta("""
SELECT *
FROM USUARIOS
WHERE APELLIDO = 'Gomez';
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL


Filas devueltas: 0


**Análisis.** La consulta no devuelve filas: no hay ningún usuario con apellido Gomez, ni tampoco con la variante Gómez. Una consulta vacía también es un resultado válido, pero la comparación con `=` es exacta, así que en un sistema real conviene buscar con `LIKE` o normalizar las tildes antes de concluir que el registro no existe.

### A7. Mostrar los estudiantes que pertenezcan al semestre 1 o al semestre 2

In [10]:
consulta("""
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE
FROM USUARIOS
WHERE ID_ROL = 1
  AND SEMESTRE IN (1, 2)
ORDER BY SEMESTRE, APELLIDO;
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
18,Elisabeth,Doppler,1
12,Cal,Stone,1
20,Mads,Nielsen,2


Filas devueltas: 3


**Análisis.** Tres estudiantes están en los primeros semestres: Cal Stone y Elisabeth Doppler en primero, y Mads Nielsen en segundo. `IN (1, 2)` equivale a `SEMESTRE = 1 OR SEMESTRE = 2` y es más fácil de leer. Los dos de primer semestre no tienen ningún curso matriculado (punto D1), un caso a revisar por la oficina de registro.

### A8. Asignaturas de la Facultad de Ingeniería y Ciencias relacionadas con inteligencia o datos

In [11]:
consulta("""
SELECT ID_CURSO, NOMBRE
FROM CURSOS
WHERE ID_FACULTAD = 1
  AND (NOMBRE LIKE '%Inteligentes%'
       OR NOMBRE LIKE '%Aprendizaje%'
       OR NOMBRE LIKE '%Datos%')
ORDER BY NOMBRE ASC;
""")

ID_CURSO,NOMBRE
20,Aprendizaje Automático
14,Comunicación de Datos
3,Estructuras de Datos
17,Gestión y Modelación de Datos
24,Sistemas Inteligentes


Filas devueltas: 5


**Análisis.** Hay 5 asignaturas de Ingeniería y Ciencias relacionadas con inteligencia o datos. Los paréntesis alrededor de las tres condiciones `LIKE` son necesarios: sin ellos, el `AND` solo aplicaría a la primera condición y aparecerían cursos de otras facultades. Desde la gestión académica llama la atención que Aprendizaje Automático y Sistemas Inteligentes están ofertados pero no tienen estudiantes (punto D4), y que Comunicación de Datos y Gestión y Modelación de Datos ni siquiera tienen horario (punto D2).

## Bloque B. Manipulación de datos (INSERT, UPDATE, DELETE)

Las sentencias de este bloque modifican `university.db`. Cada inserción se elimina al final del bloque, así que la base queda en su estado original.

### B1. Insertar una nueva facultad en la tabla FACULTADES

In [12]:
ejecutar("""
INSERT INTO FACULTADES (NOMBRE)
VALUES ('Ciencias de la Salud');
""")

consulta("""
SELECT *
FROM FACULTADES;
""")

Filas afectadas: 1
ID asignado al nuevo registro: 3


ID_FACULTAD,NOMBRE
1,Ingeniería y Ciencias
2,Humanidades y Ciencias Sociales
3,Ciencias de la Salud


Filas devueltas: 3


**Análisis.** La facultad se insertó sin indicar el ID: como `ID_FACULTAD` es `INTEGER PRIMARY KEY`, SQLite le asignó el siguiente valor disponible (3). La consulta posterior confirma que la tabla pasó de 2 a 3 facultades.

### B2. Eliminar la facultad recién insertada

In [13]:
ejecutar("""
DELETE FROM FACULTADES
WHERE NOMBRE = 'Ciencias de la Salud';
""")

Filas afectadas: 1


**Análisis.** El `DELETE` afectó exactamente una fila. Se filtra por el nombre de la facultad de prueba para borrar solo ese registro.

### B3. Verificar que la facultad fue eliminada correctamente

In [14]:
consulta("""
SELECT *
FROM FACULTADES;
""")

ID_FACULTAD,NOMBRE
1,Ingeniería y Ciencias
2,Humanidades y Ciencias Sociales


Filas devueltas: 2


**Análisis.** La tabla vuelve a tener solo las 2 facultades originales, así que la eliminación fue correcta.

### B4. Insertar un nuevo estudiante en la tabla USUARIOS

El rol de estudiante corresponde a `ID_ROL = 1` según la tabla ROLES.

In [15]:
ejecutar("""
INSERT INTO USUARIOS (NOMBRE, APELLIDO, SEMESTRE, ID_ROL)
VALUES ('Laura', 'Rojas', 1, 1);
""")

Filas afectadas: 1
ID asignado al nuevo registro: 21


**Análisis.** El nuevo estudiante quedó con `ID_USUARIO = 21` porque la tabla tenía 20 usuarios. El taller menciona el ID 22, pero el valor real depende de los registros existentes; por eso se toma el ID que reporta el motor.

### B5. Verificar que el nuevo estudiante se guardó correctamente

In [16]:
consulta("""
SELECT u.*, r.NOMBRE AS ROL
FROM USUARIOS u
INNER JOIN ROLES r ON r.ID_ROL = u.ID_ROL
WHERE u.NOMBRE = 'Laura'
  AND u.APELLIDO = 'Rojas';
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL,ROL
21,Laura,Rojas,1,1,Estudiante


Filas devueltas: 1


**Análisis.** El registro existe con rol Estudiante y semestre 1. Unir con ROLES permite comprobar que el `ID_ROL` corresponde al rol esperado y no solo que la fila exista.

### B6. Actualizar a semestre 4 el estudiante insertado

El PDF sugiere `ID_USUARIO = 22`, pero el motor asignó el 21 al nuevo registro (punto B4), así que se usa ese ID.

In [17]:
ejecutar("""
UPDATE USUARIOS
SET SEMESTRE = 4
WHERE ID_USUARIO = 21;
""")

consulta("""
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE
FROM USUARIOS
WHERE ID_USUARIO = 21;
""")

Filas afectadas: 1


ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
21,Laura,Rojas,4


Filas devueltas: 1


**Análisis.** El `UPDATE` afectó una fila y el estudiante quedó en semestre 4. Filtrar por la llave primaria garantiza que no se modifique ningún otro registro; un `UPDATE` sin `WHERE` cambiaría el semestre de todos los usuarios.

### B7. Eliminar el registro del estudiante insertado en el punto B4

In [18]:
ejecutar("""
DELETE FROM USUARIOS
WHERE ID_USUARIO = 21;
""")

Filas afectadas: 1


**Análisis.** Se eliminó una fila, la del estudiante de prueba.

### B8. Verificar que el registro del estudiante fue eliminado

In [19]:
consulta("""
SELECT COUNT(*) AS REGISTROS_CON_ID_21,
       (SELECT COUNT(*) FROM USUARIOS) AS TOTAL_USUARIOS
FROM USUARIOS
WHERE ID_USUARIO = 21;
""")

REGISTROS_CON_ID_21,TOTAL_USUARIOS
0,20


Filas devueltas: 1


**Análisis.** Ya no existe ningún usuario con ID 21 y la tabla vuelve a tener 20 usuarios, igual que al inicio. Con esto el Bloque B deja la base en su estado original.

## Bloque C. Cruces relacionales y carteleras (INNER JOIN)

**Preparación.** Las horas de `CALENDARIO_CURSOS` están guardadas como texto en formato de 12 horas (`08:00 AM`, `02:00 PM`). Si se ordenan o comparan como texto, `02:00 PM` queda antes que `08:00 AM`. Para ordenar las carteleras correctamente se crea una vista temporal que convierte cada hora a minutos desde la medianoche y asigna un número a cada día. La vista es `TEMP`, así que vive solo en esta conexión y no modifica la base.

In [20]:
conn.execute("""
CREATE TEMP VIEW IF NOT EXISTS V_HORARIO AS
SELECT
    cc.*,
    ((CAST(substr(cc.HORA_INICIO, 1, 2) AS INTEGER) % 12)
        + CASE WHEN UPPER(cc.HORA_INICIO) LIKE '%PM' THEN 12 ELSE 0 END) * 60
        + CAST(substr(cc.HORA_INICIO, 4, 2) AS INTEGER) AS INICIO_MIN,
    ((CAST(substr(cc.HORA_FIN, 1, 2) AS INTEGER) % 12)
        + CASE WHEN UPPER(cc.HORA_FIN) LIKE '%PM' THEN 12 ELSE 0 END) * 60
        + CAST(substr(cc.HORA_FIN, 4, 2) AS INTEGER) AS FIN_MIN,
    CASE cc.DIA
        WHEN 'Lunes' THEN 1
        WHEN 'Martes' THEN 2
        WHEN 'Miércoles' THEN 3
        WHEN 'Jueves' THEN 4
        WHEN 'Viernes' THEN 5
    END AS DIA_NUM
FROM CALENDARIO_CURSOS cc;
""")

consulta("""
SELECT DISTINCT HORA_INICIO, HORA_FIN, INICIO_MIN, FIN_MIN
FROM V_HORARIO
ORDER BY INICIO_MIN, FIN_MIN;
""")

HORA_INICIO,HORA_FIN,INICIO_MIN,FIN_MIN
08:00 AM,10:00 AM,480,600
09:00 AM,12:00 PM,540,720
10:00 AM,12:00 PM,600,720
02:00 PM,04:00 PM,840,960
02:00 PM,04:00PM,840,960
02:00 PM,05:00 PM,840,1020
04:00 PM,06:00 PM,960,1080
06:00 PM,09:00 PM,1080,1260


Filas devueltas: 8


**Análisis.** Hay 8 combinaciones de horario. La conversión a minutos (por ejemplo 08:00 AM = 480 y 02:00 PM = 840) permite ordenar y comparar las horas correctamente. Además aparece una inconsistencia de formato: una sesión registra la hora de fin como `04:00PM`, sin espacio. La conversión la interpreta igual que `04:00 PM` (960 minutos), pero una comparación de texto las trataría como valores distintos.

### C1. Mostrar cada carrera junto con el nombre de su facultad

In [21]:
consulta("""
SELECT c.ID_CARRERA,
       c.NOMBRE AS CARRERA,
       f.NOMBRE AS FACULTAD
FROM CARRERAS c
INNER JOIN FACULTADES f ON f.ID_FACULTAD = c.ID_FACULTAD
ORDER BY f.NOMBRE, c.NOMBRE;
""")

ID_CARRERA,CARRERA,FACULTAD
4,Filosofía,Humanidades y Ciencias Sociales
2,Ingeniería Mecánica,Ingeniería y Ciencias
1,Ingeniería de Sistemas,Ingeniería y Ciencias
3,Matemáticas Aplicadas,Ingeniería y Ciencias


Filas devueltas: 4


**Análisis.** Ingeniería y Ciencias agrupa 3 de las 4 carreras (Sistemas, Mecánica y Matemáticas Aplicadas) y Humanidades y Ciencias Sociales solo una (Filosofía). La oferta de programas está concentrada en una facultad.

### C2. Mostrar el nombre de los usuarios que tienen rol de estudiante

In [22]:
consulta("""
SELECT u.ID_USUARIO, u.NOMBRE, u.APELLIDO
FROM USUARIOS u
INNER JOIN ROLES r ON r.ID_ROL = u.ID_ROL
WHERE r.NOMBRE = 'Estudiante'
ORDER BY u.APELLIDO, u.NOMBRE;
""")

ID_USUARIO,NOMBRE,APELLIDO
18,Elisabeth,Doppler
17,Franziska,Doppler
14,Jonas,Kahnwald
13,Angelina,Meyer
20,Mads,Nielsen
19,Magnus,Nielsen
15,Martha,Nielsen
12,Cal,Stone
11,Olive,Stone
16,Bartosz,Tiedemann


Filas devueltas: 10


**Análisis.** Hay 10 estudiantes. El filtro usa el nombre del rol a través del `JOIN` con ROLES en lugar del número 1, lo que hace la consulta más legible y la protege ante cambios en la numeración de los roles.

### C3. Mostrar el nombre de los usuarios que tienen rol de profesor

In [23]:
consulta("""
SELECT u.ID_USUARIO, u.NOMBRE, u.APELLIDO
FROM USUARIOS u
INNER JOIN ROLES r ON r.ID_ROL = u.ID_ROL
WHERE r.NOMBRE = 'Profesor'
ORDER BY u.APELLIDO, u.NOMBRE;
""")

ID_USUARIO,NOMBRE,APELLIDO
6,Saanvi,Bahl
8,Charlotte,Doppler
5,Zeke,Landon
7,Ulrich,Nielsen
1,Ben,Stone
2,Grace,Stone
3,Mikaela,Stone
9,Claudia,Tiedemann
10,Robert,Vance
4,Jared,Vásquez


Filas devueltas: 10


**Análisis.** También hay 10 profesores, es decir, un profesor por cada estudiante, una proporción muy alta frente a una universidad real que confirma que se trata de una base de práctica.

### C4. Mostrar los cursos matriculados por cada estudiante

Se usa `DISTINCT` porque un mismo curso se dicta en varias sesiones semanales (por ejemplo martes y jueves) y el estudiante queda inscrito en cada una.

In [24]:
consulta("""
SELECT DISTINCT
       u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
       cu.NOMBRE AS CURSO
FROM CURSOS_ESTUDIANTES ce
INNER JOIN USUARIOS u           ON u.ID_USUARIO = ce.ID_USUARIO
INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
INNER JOIN CURSOS cu            ON cu.ID_CURSO = cc.ID_CURSO
ORDER BY ESTUDIANTE, CURSO;
""")

ESTUDIANTE,CURSO
Angelina Meyer,Filosofía Antigua
Angelina Meyer,Teología
Bartosz Tiedemann,Cinemática y Dinámica
Bartosz Tiedemann,Cálculo Diferencial
Bartosz Tiedemann,Cálculo Integral
Bartosz Tiedemann,Cálculo Multivariado
Bartosz Tiedemann,Dibujo Mecánico
Bartosz Tiedemann,Electricidad y Magnetismo
Bartosz Tiedemann,Física Térmica y Ondulatoria
Bartosz Tiedemann,Mecánica de Sólidos


Filas devueltas: 73


**Análisis.** Se obtienen 73 combinaciones de estudiante y curso para los 8 estudiantes con matrícula. Sin `DISTINCT` aparecerían 151 filas, una por cada sesión inscrita. Bartosz Tiedemann y Jonas Kahnwald tienen exactamente los mismos 13 cursos.

### C5. Mostrar los cursos programados con el salón y el edificio donde se dictan

In [25]:
consulta("""
SELECT h.ID_CALENDARIO,
       cu.NOMBRE AS CURSO,
       h.DIA,
       h.HORA_INICIO,
       h.HORA_FIN,
       s.NOMBRE AS SALON,
       e.NOMBRE AS EDIFICIO
FROM V_HORARIO h
INNER JOIN CURSOS cu    ON cu.ID_CURSO = h.ID_CURSO
INNER JOIN SALONES s    ON s.ID_SALON = h.ID_SALON
INNER JOIN EDIFICIOS e  ON e.ID_EDIFICIO = s.ID_EDIFICIO
ORDER BY cu.NOMBRE, h.DIA_NUM, h.INICIO_MIN;
""")

ID_CALENDARIO,CURSO,DIA,HORA_INICIO,HORA_FIN,SALON,EDIFICIO
33,Antropología Filosófica,Martes,08:00 AM,10:00 AM,3.6,Almendros
67,Aprendizaje Automático,Martes,04:00 PM,06:00 PM,1.1,El Samán
68,Aprendizaje Automático,Jueves,04:00 PM,06:00 PM,1.1,El Samán
11,Arquitectura del Computador,Lunes,04:00 PM,06:00 PM,2.6,El Lago
12,Arquitectura del Computador,Viernes,04:00 PM,06:00 PM,2.6,El Lago
35,Cinemática y Dinámica,Lunes,02:00 PM,04:00 PM,2.2,El Lago
36,Cinemática y Dinámica,Miércoles,02:00 PM,04:00 PM,2.2,El Lago
37,Cinemática y Dinámica,Viernes,02:00 PM,04:00 PM,2.2,El Lago
13,Cálculo Diferencial,Martes,08:00 AM,10:00 AM,2.5,El Lago
14,Cálculo Diferencial,Jueves,08:00 AM,10:00 AM,2.5,El Lago


Filas devueltas: 72


**Análisis.** Las 72 sesiones programadas se reparten en 5 edificios, pero El Lago concentra 42 (58,3 %), coherente con que es el edificio con más salones (punto E9). Cada curso tiene entre 1 y 4 sesiones semanales, normalmente en el mismo salón y a la misma hora.

### C6. Cartelera de programación del día Martes

In [26]:
consulta("""
SELECT cu.NOMBRE AS CURSO,
       h.HORA_INICIO,
       h.HORA_FIN,
       s.NOMBRE AS SALON,
       e.NOMBRE AS EDIFICIO,
       p.NOMBRE || ' ' || p.APELLIDO AS PROFESOR
FROM V_HORARIO h
INNER JOIN CURSOS cu    ON cu.ID_CURSO = h.ID_CURSO
INNER JOIN SALONES s    ON s.ID_SALON = h.ID_SALON
INNER JOIN EDIFICIOS e  ON e.ID_EDIFICIO = s.ID_EDIFICIO
INNER JOIN USUARIOS p   ON p.ID_USUARIO = h.ID_PROFESOR
WHERE h.DIA = 'Martes'
ORDER BY h.INICIO_MIN, cu.NOMBRE;
""")

CURSO,HORA_INICIO,HORA_FIN,SALON,EDIFICIO,PROFESOR
Antropología Filosófica,08:00 AM,10:00 AM,3.6,Almendros,Zeke Landon
Cálculo Diferencial,08:00 AM,10:00 AM,2.5,El Lago,Grace Stone
Introducción a la Programación,08:00 AM,10:00 AM,1.4,El Lago,Ben Stone
Programación Paralela,08:00 AM,10:00 AM,3.3,Palmas,Ulrich Nielsen
Química y Ciencia de Materiales,08:00 AM,10:00 AM,3.6,El Lago,Claudia Tiedemann
Mecánica de Sólidos,09:00 AM,12:00 PM,2.3,El Lago,Saanvi Bahl
Cálculo Integral,10:00 AM,12:00 PM,3.3,El Lago,Jared Vásquez
Estructuras de Datos,10:00 AM,12:00 PM,3.4,Almendros,Ben Stone
Álgebra Lineal,10:00 AM,12:00 PM,3.1,El Lago,Grace Stone
Cálculo Multivariado,02:00 PM,04:00 PM,2.1,El Lago,Jared Vásquez


Filas devueltas: 16


**Análisis.** El martes hay 16 sesiones, con la mayor concentración a las 08:00 AM (5 cursos simultáneos). Ordenar por `INICIO_MIN` deja la cartelera en orden cronológico; si se ordenara por el texto de `HORA_INICIO`, las sesiones de las 02:00 PM aparecerían antes que las de las 08:00 AM. La sesión de Transferencia de Calor es la que tiene la hora de fin mal digitada (`04:00PM`). Ben Stone, Claudia Tiedemann y Saanvi Bahl dictan tres sesiones cada uno ese día.

### C7. Horario completo de un estudiante específico (Mads Nielsen, ID 20)

In [27]:
consulta("""
SELECT h.DIA,
       h.HORA_INICIO,
       h.HORA_FIN,
       cu.NOMBRE AS CURSO,
       s.NOMBRE AS SALON,
       e.NOMBRE AS EDIFICIO
FROM CURSOS_ESTUDIANTES ce
INNER JOIN V_HORARIO h  ON h.ID_CALENDARIO = ce.ID_CALENDARIO
INNER JOIN CURSOS cu    ON cu.ID_CURSO = h.ID_CURSO
INNER JOIN SALONES s    ON s.ID_SALON = h.ID_SALON
INNER JOIN EDIFICIOS e  ON e.ID_EDIFICIO = s.ID_EDIFICIO
WHERE ce.ID_USUARIO = 20
ORDER BY h.DIA_NUM, h.INICIO_MIN;
""")

DIA,HORA_INICIO,HORA_FIN,CURSO,SALON,EDIFICIO
Martes,08:00 AM,10:00 AM,Introducción a la Programación,1.4,El Lago
Martes,08:00 AM,10:00 AM,Cálculo Diferencial,2.5,El Lago
Miércoles,08:00 AM,10:00 AM,Optimización Matemática,1.3,El Samán
Jueves,08:00 AM,10:00 AM,Cálculo Diferencial,2.5,El Lago
Viernes,08:00 AM,10:00 AM,Optimización Matemática,1.3,El Samán


Filas devueltas: 5


**Análisis.** Mads Nielsen tiene 5 sesiones, todas a las 08:00 AM. El martes está inscrito al mismo tiempo en Introducción a la Programación y en Cálculo Diferencial, un cruce de horario que se analiza en el Reto 2. Además solo está inscrito en la sesión del martes de Introducción a la Programación y no en la del jueves, lo que sugiere una matrícula incompleta.

### C8. Profesores registrados frente a profesores con cursos ofertados en el calendario

In [28]:
consulta("""
SELECT p.ID_USUARIO,
       p.NOMBRE || ' ' || p.APELLIDO AS PROFESOR,
       COUNT(cc.ID_CALENDARIO) AS SESIONES_OFERTADAS,
       CASE WHEN COUNT(cc.ID_CALENDARIO) > 0 THEN 'Sí' ELSE 'No' END AS TIENE_CURSOS_EN_CALENDARIO
FROM USUARIOS p
INNER JOIN ROLES r            ON r.ID_ROL = p.ID_ROL
LEFT JOIN CALENDARIO_CURSOS cc ON cc.ID_PROFESOR = p.ID_USUARIO
WHERE r.NOMBRE = 'Profesor'
GROUP BY p.ID_USUARIO
ORDER BY p.ID_USUARIO;
""")

consulta("""
SELECT (SELECT COUNT(*) FROM USUARIOS WHERE ID_ROL = 2)                  AS PROFESORES_REGISTRADOS,
       (SELECT COUNT(DISTINCT ID_PROFESOR) FROM CALENDARIO_CURSOS)       AS PROFESORES_CON_CURSOS_OFERTADOS;
""")

ID_USUARIO,PROFESOR,SESIONES_OFERTADAS,TIENE_CURSOS_EN_CALENDARIO
1,Ben Stone,8,Sí
2,Grace Stone,8,Sí
3,Mikaela Stone,6,Sí
4,Jared Vásquez,8,Sí
5,Zeke Landon,4,Sí
6,Saanvi Bahl,18,Sí
7,Ulrich Nielsen,4,Sí
8,Charlotte Doppler,2,Sí
9,Claudia Tiedemann,8,Sí
10,Robert Vance,6,Sí


Filas devueltas: 10


PROFESORES_REGISTRADOS,PROFESORES_CON_CURSOS_OFERTADOS
10,10


Filas devueltas: 1


**Análisis.** Los 10 profesores registrados tienen cursos en el calendario, así que no hay docentes sin asignación. La carga es desigual: Saanvi Bahl tiene 18 sesiones semanales, mientras que Charlotte Doppler tiene 2.

### C9. Profesores con los nombres de los cursos ofertados que orientan

In [29]:
consulta("""
SELECT DISTINCT
       p.NOMBRE || ' ' || p.APELLIDO AS PROFESOR,
       cu.NOMBRE AS CURSO
FROM CALENDARIO_CURSOS cc
INNER JOIN USUARIOS p ON p.ID_USUARIO = cc.ID_PROFESOR
INNER JOIN CURSOS cu  ON cu.ID_CURSO = cc.ID_CURSO
ORDER BY PROFESOR, CURSO;
""")

PROFESOR,CURSO
Ben Stone,Desarrollo de Videouegos
Ben Stone,Estructuras de Datos
Ben Stone,Introducción a la Programación
Ben Stone,Técnicas y Prácticas de Programación
Charlotte Doppler,Dibujo Mecánico
Claudia Tiedemann,Diseño Mecánico
Claudia Tiedemann,Máquinas Térmicas e Hidráulicas
Claudia Tiedemann,Propiedades de los Materiales
Claudia Tiedemann,Química y Ciencia de Materiales
Claudia Tiedemann,Álgebra Moderna


Filas devueltas: 40


**Análisis.** Hay 40 combinaciones de profesor y curso. Cuatro cursos los orientan dos profesores (Máquinas Térmicas e Hidráulicas, Álgebra Moderna, Optimización Matemática y Ética), lo que indica grupos distintos del mismo curso. Saanvi Bahl concentra 8 cursos, todos del área de mecánica y física.

## Bloque D. Auditoría de registros huérfanos (LEFT JOIN y subconsultas)

### D1. Estudiantes registrados sin ningún curso matriculado

In [30]:
consulta("""
SELECT u.ID_USUARIO, u.NOMBRE, u.APELLIDO, u.SEMESTRE
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_USUARIO = u.ID_USUARIO
WHERE u.ID_ROL = 1
  AND ce.ID_USUARIO IS NULL;
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
12,Cal,Stone,1
18,Elisabeth,Doppler,1


Filas devueltas: 2


**Análisis.** Cal Stone y Elisabeth Doppler, ambos de primer semestre, están registrados y vinculados a una carrera, pero no tienen ningún curso inscrito. El `LEFT JOIN` conserva a todos los estudiantes y el filtro `IS NULL` deja solo a los que no tienen pareja en CURSOS_ESTUDIANTES. Desde la gestión son casos para contactar: pueden ser estudiantes nuevos que no completaron la matrícula o estudiantes en riesgo de deserción.

### D2. Cursos del catálogo sin ningún horario programado

In [31]:
consulta("""
SELECT c.ID_CURSO, c.NOMBRE
FROM CURSOS c
LEFT JOIN CALENDARIO_CURSOS cc ON cc.ID_CURSO = c.ID_CURSO
WHERE cc.ID_CURSO IS NULL
ORDER BY c.ID_CURSO;
""")

ID_CURSO,NOMBRE
2,Introducción al Modelado de Sistemas
5,Programación Orientada a Objetos
7,Programación Funcional
8,Lógica para Ciencias de la Computación
9,Árboles y Grafos
11,Diseño de Interfaces Humano-Computador
12,Computabilidad y Complejidad
13,Computación Gráfica
14,Comunicación de Datos
15,Desarrollo Formal de Sistemas


Filas devueltas: 35


**Análisis.** 35 de los 71 cursos del catálogo (49,3 %) no tienen ningún horario. Entre ellos hay cursos centrales como Programación Orientada a Objetos, Sistemas Operativos y Ecuaciones Diferenciales, 10 cursos de Humanidades y Ciencias Sociales y Trabajo de Grado. Puede ser normal si se ofertan en otro periodo, pero conviene contrastarlo con la malla curricular de cada carrera.

### D3. Cursos del sistema sin ningún estudiante matriculado

In [32]:
consulta("""
SELECT c.ID_CURSO, c.NOMBRE
FROM CURSOS c
WHERE NOT EXISTS (
    SELECT 1
    FROM CALENDARIO_CURSOS cc
    INNER JOIN CURSOS_ESTUDIANTES ce ON ce.ID_CALENDARIO = cc.ID_CALENDARIO
    WHERE cc.ID_CURSO = c.ID_CURSO
)
ORDER BY c.ID_CURSO;
""")

ID_CURSO,NOMBRE
2,Introducción al Modelado de Sistemas
5,Programación Orientada a Objetos
7,Programación Funcional
8,Lógica para Ciencias de la Computación
9,Árboles y Grafos
11,Diseño de Interfaces Humano-Computador
12,Computabilidad y Complejidad
13,Computación Gráfica
14,Comunicación de Datos
15,Desarrollo Formal de Sistemas


Filas devueltas: 48


**Análisis.** 48 cursos (67,6 %) no tienen ningún estudiante. Incluye los 35 sin horario, que no pueden tener inscritos, y 13 que sí están programados pero que nadie tomó (punto D4). Se usa `NOT EXISTS` en lugar de `NOT IN` porque se comporta bien aunque la subconsulta devuelva valores nulos.

### D4. Cursos ofertados en el calendario sin estudiantes matriculados

In [33]:
consulta("""
SELECT c.ID_CURSO,
       c.NOMBRE,
       COUNT(cc.ID_CALENDARIO) AS SESIONES_OFERTADAS
FROM CURSOS c
INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CURSO = c.ID_CURSO
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_CALENDARIO = cc.ID_CALENDARIO
GROUP BY c.ID_CURSO
HAVING COUNT(ce.ID_USUARIO) = 0
ORDER BY c.ID_CURSO;
""")

ID_CURSO,NOMBRE,SESIONES_OFERTADAS
19,Programación Paralela,2
20,Aprendizaje Automático,2
22,Introducción a la Seguridad Informática,2
24,Sistemas Inteligentes,2
26,Desarrollo de Videouegos,2
38,Transferencia de Calor,2
39,Diseño Mecánico,2
40,Máquinas Térmicas e Hidráulicas,3
52,Estadítica Aplicada,1
65,Antropología Filosófica,1


Filas devueltas: 13


**Análisis.** 13 cursos tienen horario, salón y profesor asignados pero ningún estudiante, y ocupan 23 sesiones semanales sin uso. Entre ellos están Aprendizaje Automático, Sistemas Inteligentes y Programación Paralela. Es el hallazgo con más impacto en costos: son candidatos a revisar su oferta, su horario o su difusión.

## Bloque E. Agregaciones, estadísticas y cupos (GROUP BY y HAVING)

### E1. Total de estudiantes de la universidad

In [34]:
consulta("""
SELECT COUNT(*) AS TOTAL_ESTUDIANTES
FROM USUARIOS u
INNER JOIN ROLES r ON r.ID_ROL = u.ID_ROL
WHERE r.NOMBRE = 'Estudiante';
""")

TOTAL_ESTUDIANTES
10


Filas devueltas: 1


**Análisis.** La universidad tiene 10 estudiantes registrados.

### E2. Total de profesores y total de estudiantes en una sola consulta

In [35]:
consulta("""
SELECT SUM(CASE WHEN ID_ROL = 2 THEN 1 ELSE 0 END) AS TOTAL_PROFESORES,
       SUM(CASE WHEN ID_ROL = 1 THEN 1 ELSE 0 END) AS TOTAL_ESTUDIANTES
FROM USUARIOS;
""")

TOTAL_PROFESORES,TOTAL_ESTUDIANTES
10,10


Filas devueltas: 1


**Análisis.** Hay 10 profesores y 10 estudiantes. Con `SUM` y `CASE` se cuentan los dos grupos en una sola lectura de la tabla, sin necesidad de dos consultas.

### E3. Promedio del semestre en el que se encuentran los estudiantes

Se filtra `ID_ROL = 1` porque los profesores tienen `SEMESTRE = 0` y bajarían el promedio.

In [36]:
consulta("""
SELECT ROUND(AVG(SEMESTRE), 2) AS PROMEDIO_SEMESTRE,
       MIN(SEMESTRE)           AS SEMESTRE_MINIMO,
       MAX(SEMESTRE)           AS SEMESTRE_MAXIMO
FROM USUARIOS
WHERE ID_ROL = 1;
""")

PROMEDIO_SEMESTRE,SEMESTRE_MINIMO,SEMESTRE_MAXIMO
4,1,6


Filas devueltas: 1


**Análisis.** El semestre promedio de los estudiantes es 4, en un rango de 1 a 6. Si no se filtrara el rol, los 10 profesores con semestre 0 bajarían el promedio a 2, un resultado engañoso.

### E4. Estudiante o estudiantes que cursan el semestre más alto

In [37]:
consulta("""
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE
FROM USUARIOS
WHERE ID_ROL = 1
  AND SEMESTRE = (SELECT MAX(SEMESTRE) FROM USUARIOS WHERE ID_ROL = 1);
""")

ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
19,Magnus,Nielsen,6


Filas devueltas: 1


**Análisis.** Magnus Nielsen, en sexto semestre, es el estudiante más avanzado. La subconsulta calcula el máximo y la consulta externa devuelve a todos los que lo alcanzan, así que también funcionaría si hubiera empates.

### E5. Cantidad de cursos ofertados por cada facultad

In [38]:
consulta("""
SELECT f.NOMBRE                    AS FACULTAD,
       COUNT(DISTINCT c.ID_CURSO)  AS CURSOS_EN_CATALOGO,
       COUNT(DISTINCT cc.ID_CURSO) AS CURSOS_OFERTADOS,
       COUNT(cc.ID_CALENDARIO)     AS SESIONES_PROGRAMADAS
FROM FACULTADES f
LEFT JOIN CURSOS c             ON c.ID_FACULTAD = f.ID_FACULTAD
LEFT JOIN CALENDARIO_CURSOS cc ON cc.ID_CURSO = c.ID_CURSO
GROUP BY f.ID_FACULTAD
ORDER BY CURSOS_OFERTADOS DESC;
""")

FACULTAD,CURSOS_EN_CATALOGO,CURSOS_OFERTADOS,SESIONES_PROGRAMADAS
Ingeniería y Ciencias,54,30,65
Humanidades y Ciencias Sociales,15,5,6


Filas devueltas: 2


**Análisis.** Ingeniería y Ciencias tiene 54 cursos en su catálogo y oferta 30 (55,6 %) en 65 sesiones; Humanidades y Ciencias Sociales tiene 15 y solo oferta 5 (33,3 %) en 6 sesiones. Los dos cursos sin facultad no aparecen en esta consulta: Práctica Estudiantil está ofertado en 1 sesión y Trabajo de Grado no tiene horario, lo que completa los 36 cursos y las 72 sesiones del calendario.

### E6. Facultades con más de 3 carreras asociadas

La segunda consulta, sin `HAVING`, muestra el conteo completo para comprobar el resultado de la primera.

In [39]:
consulta("""
SELECT f.NOMBRE AS FACULTAD,
       COUNT(c.ID_CARRERA) AS CARRERAS
FROM FACULTADES f
INNER JOIN CARRERAS c ON c.ID_FACULTAD = f.ID_FACULTAD
GROUP BY f.ID_FACULTAD
HAVING COUNT(c.ID_CARRERA) > 3;
""")

consulta("""
SELECT f.NOMBRE AS FACULTAD,
       COUNT(c.ID_CARRERA) AS CARRERAS
FROM FACULTADES f
LEFT JOIN CARRERAS c ON c.ID_FACULTAD = f.ID_FACULTAD
GROUP BY f.ID_FACULTAD
ORDER BY CARRERAS DESC;
""")

FACULTAD,CARRERAS


Filas devueltas: 0


FACULTAD,CARRERAS
Ingeniería y Ciencias,3
Humanidades y Ciencias Sociales,1


Filas devueltas: 2


**Análisis.** Ninguna facultad tiene más de 3 carreras, así que la consulta con `HAVING` no devuelve filas. El conteo completo lo confirma: Ingeniería y Ciencias tiene exactamente 3 y Humanidades y Ciencias Sociales 1. `HAVING` filtra los grupos después de agregarlos, a diferencia de `WHERE`, que filtra filas antes de agrupar.

### E7. Cantidad de estudiantes matriculados por cada carrera

In [40]:
consulta("""
SELECT ca.NOMBRE AS CARRERA,
       COUNT(ce.ID_USUARIO) AS ESTUDIANTES
FROM CARRERAS ca
LEFT JOIN CARRERAS_ESTUDIANTES ce ON ce.ID_CARRERA = ca.ID_CARRERA
GROUP BY ca.ID_CARRERA
ORDER BY ESTUDIANTES DESC, CARRERA;
""")

CARRERA,ESTUDIANTES
Ingeniería Mecánica,4
Ingeniería de Sistemas,3
Matemáticas Aplicadas,2
Filosofía,1


Filas devueltas: 4


**Análisis.** Ingeniería Mecánica tiene 4 estudiantes, Ingeniería de Sistemas 3, Matemáticas Aplicadas 2 y Filosofía 1, para un total de 10: cada estudiante está en una sola carrera. El `LEFT JOIN` asegura que una carrera sin estudiantes aparezca con 0 en lugar de desaparecer.

### E8. Carrera con mayor cantidad de estudiantes matriculados

Se compara contra el máximo en lugar de usar `LIMIT 1` para no perder empates.

In [41]:
consulta("""
WITH conteo AS (
    SELECT ca.NOMBRE AS CARRERA,
           COUNT(ce.ID_USUARIO) AS ESTUDIANTES
    FROM CARRERAS ca
    LEFT JOIN CARRERAS_ESTUDIANTES ce ON ce.ID_CARRERA = ca.ID_CARRERA
    GROUP BY ca.ID_CARRERA
)
SELECT CARRERA, ESTUDIANTES
FROM conteo
WHERE ESTUDIANTES = (SELECT MAX(ESTUDIANTES) FROM conteo);
""")

CARRERA,ESTUDIANTES
Ingeniería Mecánica,4


Filas devueltas: 1


**Análisis.** Ingeniería Mecánica es la carrera con más estudiantes (4).

### E9. Edificio con mayor cantidad de salones registrados

In [42]:
consulta("""
WITH conteo AS (
    SELECT e.NOMBRE AS EDIFICIO,
           COUNT(s.ID_SALON) AS SALONES
    FROM EDIFICIOS e
    LEFT JOIN SALONES s ON s.ID_EDIFICIO = e.ID_EDIFICIO
    GROUP BY e.ID_EDIFICIO
)
SELECT EDIFICIO, SALONES
FROM conteo
WHERE SALONES = (SELECT MAX(SALONES) FROM conteo);
""")

EDIFICIO,SALONES
El Lago,18


Filas devueltas: 1


**Análisis.** El Lago tiene 18 de los 42 salones (42,9 %), seguido por El Samán y Almendros con 8 cada uno.

### E10. Cursos ofertados por cada docente y profesores con mayor carga

In [43]:
consulta("""
WITH carga AS (
    SELECT p.ID_USUARIO,
           p.NOMBRE || ' ' || p.APELLIDO AS PROFESOR,
           COUNT(DISTINCT cc.ID_CURSO)   AS CURSOS,
           COUNT(cc.ID_CALENDARIO)       AS SESIONES
    FROM USUARIOS p
    INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_PROFESOR = p.ID_USUARIO
    GROUP BY p.ID_USUARIO
)
SELECT PROFESOR,
       CURSOS,
       SESIONES,
       CASE WHEN CURSOS = (SELECT MAX(CURSOS) FROM carga) THEN 'Mayor carga' ELSE '' END AS OBSERVACION
FROM carga
ORDER BY CURSOS DESC, SESIONES DESC, PROFESOR;
""")

PROFESOR,CURSOS,SESIONES,OBSERVACION
Saanvi Bahl,8,18,Mayor carga
Claudia Tiedemann,5,8,
Jared Vásquez,5,8,
Ben Stone,4,8,
Grace Stone,4,8,
Mikaela Stone,4,6,
Zeke Landon,4,4,
Robert Vance,3,6,
Ulrich Nielsen,2,4,
Charlotte Doppler,1,2,


Filas devueltas: 10


**Análisis.** Saanvi Bahl orienta 8 cursos en 18 sesiones semanales, más del doble de sesiones que los siguientes (Claudia Tiedemann y Jared Vásquez, con 5 cursos y 8 sesiones). En el otro extremo, Charlotte Doppler tiene 1 curso. Esta concentración es un riesgo operativo: una ausencia de la profesora con mayor carga afectaría 8 cursos.

### E11. Número de matrículas realizadas por cada estudiante

Cada matrícula es una inscripción a una sesión del calendario (`CURSOS_ESTUDIANTES`).

In [44]:
consulta("""
SELECT u.ID_USUARIO,
       u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
       COUNT(ce.ID_CALENDARIO)       AS MATRICULAS
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_USUARIO = u.ID_USUARIO
WHERE u.ID_ROL = 1
GROUP BY u.ID_USUARIO
ORDER BY MATRICULAS DESC, ESTUDIANTE;
""")

ID_USUARIO,ESTUDIANTE,MATRICULAS
19,Magnus Nielsen,29
16,Bartosz Tiedemann,28
14,Jonas Kahnwald,28
17,Franziska Doppler,21
15,Martha Nielsen,19
11,Olive Stone,19
20,Mads Nielsen,5
13,Angelina Meyer,2
12,Cal Stone,0
18,Elisabeth Doppler,0


Filas devueltas: 10


**Análisis.** Las matrículas están muy concentradas: Magnus Nielsen (29), Bartosz Tiedemann (28) y Jonas Kahnwald (28) suman 85 de las 151 inscripciones (56,3 %), mientras que Angelina Meyer tiene 2 y dos estudiantes no tienen ninguna.

### E12. Estudiantes matriculados en menos de 10 cursos

Se cuentan cursos distintos (no sesiones) y se incluyen con 0 los estudiantes sin matrícula.

In [45]:
consulta("""
SELECT u.ID_USUARIO,
       u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
       COUNT(DISTINCT cc.ID_CURSO)   AS CURSOS,
       COUNT(ce.ID_CALENDARIO)       AS MATRICULAS
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_USUARIO = u.ID_USUARIO
LEFT JOIN CALENDARIO_CURSOS cc  ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
WHERE u.ID_ROL = 1
GROUP BY u.ID_USUARIO
HAVING COUNT(DISTINCT cc.ID_CURSO) < 10
ORDER BY CURSOS, ESTUDIANTE;
""")

ID_USUARIO,ESTUDIANTE,CURSOS,MATRICULAS
12,Cal Stone,0,0
18,Elisabeth Doppler,0,0
13,Angelina Meyer,2,2
20,Mads Nielsen,3,5
15,Martha Nielsen,9,19
11,Olive Stone,9,19


Filas devueltas: 6


**Análisis.** Seis estudiantes tienen menos de 10 cursos. Contar cursos distintos y no inscripciones cambia el resultado: Martha Nielsen y Olive Stone tienen 19 inscripciones pero solo 9 cursos, porque cada curso tiene 2 o 3 sesiones semanales. Cal Stone y Elisabeth Doppler aparecen con 0 gracias al `LEFT JOIN`.

### E13. Curso con más inscripciones y curso con más estudiantes únicos

In [46]:
consulta("""
WITH por_curso AS (
    SELECT cu.NOMBRE                     AS CURSO,
           COUNT(*)                      AS INSCRIPCIONES,
           COUNT(DISTINCT ce.ID_USUARIO) AS ESTUDIANTES_UNICOS
    FROM CURSOS_ESTUDIANTES ce
    INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
    INNER JOIN CURSOS cu            ON cu.ID_CURSO = cc.ID_CURSO
    GROUP BY cu.ID_CURSO
)
SELECT 'Mayor número de inscripciones' AS CRITERIO, CURSO, INSCRIPCIONES, ESTUDIANTES_UNICOS
FROM por_curso
WHERE INSCRIPCIONES = (SELECT MAX(INSCRIPCIONES) FROM por_curso)
UNION ALL
SELECT 'Mayor número de estudiantes únicos', CURSO, INSCRIPCIONES, ESTUDIANTES_UNICOS
FROM por_curso
WHERE ESTUDIANTES_UNICOS = (SELECT MAX(ESTUDIANTES_UNICOS) FROM por_curso);
""")

CRITERIO,CURSO,INSCRIPCIONES,ESTUDIANTES_UNICOS
Mayor número de inscripciones,Cinemática y Dinámica,18,6
Mayor número de estudiantes únicos,Cálculo Diferencial,14,7


Filas devueltas: 2


**Análisis.** Cinemática y Dinámica tiene más inscripciones (18) pero solo 6 estudiantes, porque se dicta en 3 sesiones semanales; Cálculo Diferencial tiene menos inscripciones (14) pero más estudiantes distintos (7). Las inscripciones miden la carga de sesiones, mientras que los estudiantes únicos miden la demanda real del curso.

### E14. Análisis de demanda y cupos: secciones con 5 o más inscritos

In [47]:
consulta("""
SELECT cc.ID_CALENDARIO,
       cu.NOMBRE AS CURSO,
       cc.DIA,
       cc.CUPO,
       COUNT(ce.ID_USUARIO) AS INSCRITOS,
       ROUND(100.0 * COUNT(ce.ID_USUARIO) / cc.CUPO, 1) AS OCUPACION_PCT
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cu             ON cu.ID_CURSO = cc.ID_CURSO
INNER JOIN CURSOS_ESTUDIANTES ce ON ce.ID_CALENDARIO = cc.ID_CALENDARIO
GROUP BY cc.ID_CALENDARIO
HAVING COUNT(ce.ID_USUARIO) >= 5
ORDER BY INSCRITOS DESC, cc.ID_CALENDARIO;
""")

ID_CALENDARIO,CURSO,DIA,CUPO,INSCRITOS,OCUPACION_PCT
13,Cálculo Diferencial,Martes,30,7,"23,3"
14,Cálculo Diferencial,Jueves,30,7,"23,3"
15,Álgebra Lineal,Martes,30,6,"20,0"
16,Álgebra Lineal,Jueves,30,6,"20,0"
23,Cálculo Integral,Martes,20,6,"30,0"
24,Cálculo Integral,Jueves,20,6,"30,0"
25,Cálculo Multivariado,Martes,20,6,"30,0"
26,Cálculo Multivariado,Jueves,20,6,"30,0"
31,Teología,Lunes,30,6,"20,0"
35,Cinemática y Dinámica,Lunes,30,6,"20,0"


Filas devueltas: 15


**Análisis.** Solo 15 de las 72 sesiones tienen 5 o más inscritos, y ninguna supera el 30 % de su cupo: las más llenas son Cálculo Integral y Cálculo Multivariado, con 6 estudiantes en cupos de 20. En total hay 151 inscripciones para 1.745 cupos, una ocupación global de 8,7 %. Hay mucha capacidad ociosa; se podrían fusionar secciones o reducir cupos sin afectar a los estudiantes.

## Parte II. Retos integradores

## Reto integrador 1. Carga académica por carrera

Para cada carrera se necesitan dos conteos que vienen de dos tablas de relación muchos a muchos (`CURSOS_CARRERAS` y `CARRERAS_ESTUDIANTES`). Si se unen las dos tablas en la misma consulta, cada curso se repite por cada estudiante y los conteos se multiplican. Por eso cada conteo se calcula por separado en una CTE y después se unen por carrera.

In [48]:
consulta("""
WITH malla AS (
    SELECT ID_CARRERA, COUNT(*) AS CURSOS_MALLA
    FROM CURSOS_CARRERAS
    GROUP BY ID_CARRERA
),
matricula AS (
    SELECT ID_CARRERA, COUNT(*) AS ESTUDIANTES
    FROM CARRERAS_ESTUDIANTES
    GROUP BY ID_CARRERA
)
SELECT ca.NOMBRE                     AS CARRERA,
       f.NOMBRE                      AS FACULTAD,
       COALESCE(m.CURSOS_MALLA, 0)   AS CURSOS_MALLA,
       COALESCE(e.ESTUDIANTES, 0)    AS ESTUDIANTES
FROM CARRERAS ca
INNER JOIN FACULTADES f ON f.ID_FACULTAD = ca.ID_FACULTAD
LEFT JOIN malla m       ON m.ID_CARRERA = ca.ID_CARRERA
LEFT JOIN matricula e   ON e.ID_CARRERA = ca.ID_CARRERA
ORDER BY CURSOS_MALLA DESC;
""")

CARRERA,FACULTAD,CURSOS_MALLA,ESTUDIANTES
Ingeniería de Sistemas,Ingeniería y Ciencias,37,3
Ingeniería Mecánica,Ingeniería y Ciencias,25,4
Matemáticas Aplicadas,Ingeniería y Ciencias,22,2
Filosofía,Humanidades y Ciencias Sociales,17,1


Filas devueltas: 4


**Análisis.** Ingeniería de Sistemas tiene la malla más grande (37 cursos) pero solo 3 estudiantes; Ingeniería Mecánica tiene 25 cursos y la mayor matrícula (4 estudiantes); Filosofía, con 17 cursos, tiene un único estudiante. Al comparar cursos de la malla con estudiantes aparecen programas con mucha oferta y poca demanda, sobre todo Sistemas (12,3 cursos por estudiante) y Filosofía (17).

**Comprobación del producto cartesiano.** La siguiente consulta une las dos tablas de relación directamente, que es el error que advierte la nota técnica del taller.

In [49]:
consulta("""
SELECT ca.NOMBRE              AS CARRERA,
       COUNT(cc.ID_CURSO)     AS CURSOS_MALLA_INFLADO,
       COUNT(ce.ID_USUARIO)   AS ESTUDIANTES_INFLADO
FROM CARRERAS ca
INNER JOIN CURSOS_CARRERAS cc      ON cc.ID_CARRERA = ca.ID_CARRERA
INNER JOIN CARRERAS_ESTUDIANTES ce ON ce.ID_CARRERA = ca.ID_CARRERA
GROUP BY ca.ID_CARRERA
ORDER BY CURSOS_MALLA_INFLADO DESC;
""")

CARRERA,CURSOS_MALLA_INFLADO,ESTUDIANTES_INFLADO
Ingeniería de Sistemas,111,111
Ingeniería Mecánica,100,100
Matemáticas Aplicadas,44,44
Filosofía,17,17


Filas devueltas: 4


**Análisis.** Al unir directamente las dos tablas de relación, cada curso de la malla se repite por cada estudiante de la carrera: Sistemas pasa de 37 cursos y 3 estudiantes a 111 en los dos conteos (37 por 3), y Mecánica a 100 (25 por 4). Solo Filosofía da el valor correcto porque tiene un único estudiante, lo que haría difícil detectar el error si se revisara solo esa fila. Por eso la consulta correcta calcula cada conteo por separado en una CTE.

## Reto integrador 2. Auditoría Text-to-SQL de cruces de horario

**Pregunta en lenguaje natural para el agente de Antigravity:** *"Identifica qué estudiantes tienen cruces de horario (dos clases el mismo día y a la misma hora) en university_db"*.

Para auditar la consulta del agente primero se construye una **consulta de referencia** verificada. Dos sesiones de un estudiante se cruzan si son el mismo día y sus intervalos se solapan, es decir, si cada una empieza antes de que termine la otra (`inicio_a < fin_b` y `inicio_b < fin_a`). La consulta de referencia cuida cuatro puntos:

1. Convierte las horas de texto (`08:00 AM`, `02:00 PM`, e incluso `04:00PM` sin espacio) a minutos, porque compararlas como texto da resultados incorrectos.
2. Detecta solapamientos parciales (por ejemplo 08:00 a 10:00 contra 09:00 a 12:00) y no solo horas de inicio iguales.
3. Usa `a.ID_CALENDARIO < b.ID_CALENDARIO` para no comparar una sesión consigo misma ni contar el mismo par dos veces.
4. No marca como cruce dos clases consecutivas (una termina a las 10:00 y la otra empieza a las 10:00).

In [50]:
sql_referencia = """
WITH horario_estudiante AS (
    SELECT ce.ID_USUARIO,
           cc.ID_CALENDARIO,
           cc.ID_CURSO,
           cc.DIA,
           cc.HORA_INICIO,
           cc.HORA_FIN,
           ((CAST(substr(cc.HORA_INICIO, 1, 2) AS INTEGER) % 12)
               + CASE WHEN UPPER(cc.HORA_INICIO) LIKE '%PM' THEN 12 ELSE 0 END) * 60
               + CAST(substr(cc.HORA_INICIO, 4, 2) AS INTEGER) AS INICIO_MIN,
           ((CAST(substr(cc.HORA_FIN, 1, 2) AS INTEGER) % 12)
               + CASE WHEN UPPER(cc.HORA_FIN) LIKE '%PM' THEN 12 ELSE 0 END) * 60
               + CAST(substr(cc.HORA_FIN, 4, 2) AS INTEGER) AS FIN_MIN,
           CASE cc.DIA
               WHEN 'Lunes' THEN 1
               WHEN 'Martes' THEN 2
               WHEN 'Miércoles' THEN 3
               WHEN 'Jueves' THEN 4
               WHEN 'Viernes' THEN 5
           END AS DIA_NUM
    FROM CURSOS_ESTUDIANTES ce
    INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
)
SELECT a.ID_USUARIO,
       u.NOMBRE || ' ' || u.APELLIDO           AS ESTUDIANTE,
       a.DIA,
       a.ID_CALENDARIO                         AS SESION_1,
       c1.NOMBRE                               AS CURSO_1,
       a.HORA_INICIO || ' a ' || a.HORA_FIN    AS HORARIO_1,
       b.ID_CALENDARIO                         AS SESION_2,
       c2.NOMBRE                               AS CURSO_2,
       b.HORA_INICIO || ' a ' || b.HORA_FIN    AS HORARIO_2
FROM horario_estudiante a
INNER JOIN horario_estudiante b
        ON  b.ID_USUARIO = a.ID_USUARIO
        AND b.DIA = a.DIA
        AND a.ID_CALENDARIO < b.ID_CALENDARIO
        AND a.INICIO_MIN < b.FIN_MIN
        AND b.INICIO_MIN < a.FIN_MIN
INNER JOIN USUARIOS u ON u.ID_USUARIO = a.ID_USUARIO
INNER JOIN CURSOS c1  ON c1.ID_CURSO = a.ID_CURSO
INNER JOIN CURSOS c2  ON c2.ID_CURSO = b.ID_CURSO
ORDER BY ESTUDIANTE, a.DIA_NUM, a.INICIO_MIN, SESION_1, SESION_2;
"""
consulta(sql_referencia)

ID_USUARIO,ESTUDIANTE,DIA,SESION_1,CURSO_1,HORARIO_1,SESION_2,CURSO_2,HORARIO_2
16,Bartosz Tiedemann,Lunes,31,Teología,10:00 AM a 12:00 PM,41,Termodinámica,10:00 AM a 12:00 PM
16,Bartosz Tiedemann,Lunes,31,Teología,10:00 AM a 12:00 PM,57,Dibujo Mecánico,10:00 AM a 12:00 PM
16,Bartosz Tiedemann,Lunes,41,Termodinámica,10:00 AM a 12:00 PM,57,Dibujo Mecánico,10:00 AM a 12:00 PM
16,Bartosz Tiedemann,Martes,13,Cálculo Diferencial,08:00 AM a 10:00 AM,47,Mecánica de Sólidos,09:00 AM a 12:00 PM
16,Bartosz Tiedemann,Martes,13,Cálculo Diferencial,08:00 AM a 10:00 AM,59,Química y Ciencia de Materiales,08:00 AM a 10:00 AM
16,Bartosz Tiedemann,Martes,47,Mecánica de Sólidos,09:00 AM a 12:00 PM,59,Química y Ciencia de Materiales,08:00 AM a 10:00 AM
16,Bartosz Tiedemann,Martes,15,Álgebra Lineal,10:00 AM a 12:00 PM,23,Cálculo Integral,10:00 AM a 12:00 PM
16,Bartosz Tiedemann,Martes,15,Álgebra Lineal,10:00 AM a 12:00 PM,47,Mecánica de Sólidos,09:00 AM a 12:00 PM
16,Bartosz Tiedemann,Martes,23,Cálculo Integral,10:00 AM a 12:00 PM,47,Mecánica de Sólidos,09:00 AM a 12:00 PM
16,Bartosz Tiedemann,Martes,25,Cálculo Multivariado,02:00 PM a 04:00 PM,61,Propiedades de los Materiales,02:00 PM a 04:00 PM


Filas devueltas: 70


**Análisis.** La consulta encuentra 70 pares de sesiones que se cruzan. La mayoría son clases a la misma hora (por ejemplo Teología, Termodinámica y Dibujo Mecánico, los lunes de 10:00 AM a 12:00 PM), pero 17 son solapamientos parciales con las sesiones de 09:00 AM a 12:00 PM de Mecánica de Sólidos y Mecánica de Fluidos, que una consulta basada solo en horas de inicio iguales no detectaría.

In [51]:
cruces = pd.read_sql_query(sql_referencia, conn)
resumen = (cruces.groupby(['ID_USUARIO', 'ESTUDIANTE'])
           .agg(CRUCES=('SESION_1', 'size'), DIAS_CON_CRUCE=('DIA', 'nunique'))
           .reset_index()
           .sort_values(['CRUCES', 'ESTUDIANTE'], ascending=[False, True]))
display(resumen.style.hide(axis='index'))
print(f"Estudiantes con al menos un cruce: {len(resumen)} | Pares de sesiones en cruce: {len(cruces)}")

ID_USUARIO,ESTUDIANTE,CRUCES,DIAS_CON_CRUCE
19,Magnus Nielsen,20,3
16,Bartosz Tiedemann,15,3
14,Jonas Kahnwald,15,3
17,Franziska Doppler,11,5
15,Martha Nielsen,4,4
11,Olive Stone,4,4
20,Mads Nielsen,1,1


Estudiantes con al menos un cruce: 7 | Pares de sesiones en cruce: 70


**Análisis.** Siete de los ocho estudiantes con matrícula tienen al menos un cruce; solo Angelina Meyer no tiene. Magnus Nielsen acumula 20 cruces en 3 días, y Bartosz Tiedemann y Jonas Kahnwald 15 cada uno, con el mismo patrón porque su matrícula es idéntica. Franziska Doppler tiene cruces los 5 días de la semana. Con este nivel de cruces los estudiantes no pueden asistir a todas sus clases: el proceso de matrícula debería validar los horarios antes de aceptar una inscripción.

### Errores típicos de una consulta Text-to-SQL

Las siguientes consultas son **ilustrativas**: reproducen errores lógicos frecuentes en consultas generadas automáticamente para esta misma pregunta. No son la respuesta del agente de Antigravity; sirven como lista de control para auditarla. Cada una devuelve el estudiante y el par de sesiones, y se compara contra la consulta de referencia.

In [52]:
errores_tipicos = {
    'Solo horas de inicio iguales': """
        SELECT e1.ID_USUARIO, c1.ID_CALENDARIO AS S1, c2.ID_CALENDARIO AS S2
        FROM CURSOS_ESTUDIANTES e1
        JOIN CALENDARIO_CURSOS c1  ON c1.ID_CALENDARIO = e1.ID_CALENDARIO
        JOIN CURSOS_ESTUDIANTES e2 ON e2.ID_USUARIO = e1.ID_USUARIO
        JOIN CALENDARIO_CURSOS c2  ON c2.ID_CALENDARIO = e2.ID_CALENDARIO
        WHERE c1.DIA = c2.DIA
          AND c1.HORA_INICIO = c2.HORA_INICIO
          AND c1.ID_CALENDARIO < c2.ID_CALENDARIO
    """,
    'Sin excluir la misma sesión': """
        SELECT e1.ID_USUARIO, h1.ID_CALENDARIO AS S1, h2.ID_CALENDARIO AS S2
        FROM CURSOS_ESTUDIANTES e1
        JOIN V_HORARIO h1          ON h1.ID_CALENDARIO = e1.ID_CALENDARIO
        JOIN CURSOS_ESTUDIANTES e2 ON e2.ID_USUARIO = e1.ID_USUARIO
        JOIN V_HORARIO h2          ON h2.ID_CALENDARIO = e2.ID_CALENDARIO
        WHERE h1.DIA = h2.DIA
          AND h1.INICIO_MIN < h2.FIN_MIN
          AND h2.INICIO_MIN < h1.FIN_MIN
    """,
    'Horas comparadas como texto': """
        SELECT e1.ID_USUARIO, c1.ID_CALENDARIO AS S1, c2.ID_CALENDARIO AS S2
        FROM CURSOS_ESTUDIANTES e1
        JOIN CALENDARIO_CURSOS c1  ON c1.ID_CALENDARIO = e1.ID_CALENDARIO
        JOIN CURSOS_ESTUDIANTES e2 ON e2.ID_USUARIO = e1.ID_USUARIO
        JOIN CALENDARIO_CURSOS c2  ON c2.ID_CALENDARIO = e2.ID_CALENDARIO
        WHERE c1.DIA = c2.DIA
          AND c1.ID_CALENDARIO < c2.ID_CALENDARIO
          AND c1.HORA_INICIO < c2.HORA_FIN
          AND c2.HORA_INICIO < c1.HORA_FIN
    """,
}

referencia = set(cruces[['ID_USUARIO', 'SESION_1', 'SESION_2']].itertuples(index=False, name=None))


def auditar(nombre, filas):
    """Compara un resultado con columnas (estudiante, sesion, sesion) contra la consulta de referencia."""
    tuplas = list(filas.itertuples(index=False, name=None))
    pares = {(u, min(s1, s2), max(s1, s2)) for u, s1, s2 in tuplas if s1 != s2}
    return {'CONSULTA': nombre,
            'FILAS_DEVUELTAS': len(tuplas),
            'AUTOCOMPARACIONES': sum(1 for _, s1, s2 in tuplas if s1 == s2),
            'PARES_DISTINTOS': len(pares),
            'CRUCES_CORRECTOS': len(pares & referencia),
            'FALSOS_POSITIVOS': len(pares - referencia),
            'CRUCES_OMITIDOS': len(referencia - pares)}


comparacion = [auditar('Consulta de referencia', cruces[['ID_USUARIO', 'SESION_1', 'SESION_2']])]
for nombre, sql in errores_tipicos.items():
    comparacion.append(auditar(nombre, pd.read_sql_query(sql, conn)))
display(pd.DataFrame(comparacion).style.hide(axis='index'))

CONSULTA,FILAS_DEVUELTAS,AUTOCOMPARACIONES,PARES_DISTINTOS,CRUCES_CORRECTOS,FALSOS_POSITIVOS,CRUCES_OMITIDOS
Consulta de referencia,70,0,70,70,0,0
Solo horas de inicio iguales,53,0,53,53,0,17
Sin excluir la misma sesión,291,151,70,70,0,0
Horas comparadas como texto,70,0,70,70,0,0


**Análisis.** La tabla mide cada error frente a la referencia. Comparar solo horas de inicio iguales encuentra 53 cruces y omite los 17 parciales. No excluir la misma sesión produce 151 autocomparaciones (cada inscripción se cruza consigo misma) y repite cada par en los dos órdenes, así que devuelve 291 filas para 70 cruces reales; cualquier conteo de cruces por estudiante quedaría inflado. La comparación de horas como texto coincide con la referencia en estos datos, pero solo por casualidad, como muestra la siguiente consulta.

In [53]:
consulta("""
SELECT '12:00 PM' < '02:00 PM' AS TEXTO_12PM_ANTES_DE_2PM,
       720 < 840               AS MINUTOS_12PM_ANTES_DE_2PM,
       '11:00 AM' < '01:00 PM' AS TEXTO_11AM_ANTES_DE_1PM,
       660 < 780               AS MINUTOS_11AM_ANTES_DE_1PM,
       '04:00PM' = '04:00 PM'  AS TEXTO_FORMATOS_IGUALES;
""")

TEXTO_12PM_ANTES_DE_2PM,MINUTOS_12PM_ANTES_DE_2PM,TEXTO_11AM_ANTES_DE_1PM,MINUTOS_11AM_ANTES_DE_1PM,TEXTO_FORMATOS_IGUALES
0,1,0,1,0


Filas devueltas: 1


**Análisis.** Como texto, `'12:00 PM'` no resulta menor que `'02:00 PM'` y `'11:00 AM'` no resulta menor que `'01:00 PM'`, aunque en minutos sí lo son, y `'04:00PM'` no es igual a `'04:00 PM'`. En university.db todos los cruces ocurren entre sesiones de la misma jornada y ninguna sesión atraviesa el mediodía, por eso el error no se manifiesta; bastaría con programar una clase de 11:00 AM a 01:00 PM para que una consulta basada en texto fallara. Una consulta confiable debe convertir las horas antes de compararlas.

### Consulta generada por el agente de Antigravity

En la siguiente celda se pega, entre las comillas triples, la consulta SQL que entregue el agente de Antigravity (panel derecho, conectado al servidor MCP de DBCode) ante la pregunta del reto. Al ejecutarla se muestra su resultado para compararlo con la consulta de referencia (70 pares de sesiones en cruce, 7 estudiantes) y con la lista de errores típicos. Si la consulta del agente devuelve el estudiante y los dos `ID_CALENDARIO` en ese orden, la función `auditar` calcula automáticamente sus aciertos, falsos positivos y cruces omitidos.

In [54]:
consulta_agente = """
"""

if consulta_agente.strip():
    resultado_agente = pd.read_sql_query(consulta_agente, conn)
    display(resultado_agente.style.hide(axis='index'))
    print(f'Filas devueltas: {len(resultado_agente)}')
else:
    print('Pendiente: pegar la consulta generada por el agente de Antigravity y volver a ejecutar esta celda.')

Pendiente: pegar la consulta generada por el agente de Antigravity y volver a ejecutar esta celda.


**Análisis crítico de la consulta del agente.** Pendiente de completar con la consulta real que genere el agente de Antigravity. Al compararla con la consulta de referencia se debe revisar: si compara una sesión consigo misma o devuelve cada par dos veces, si detecta solapamientos parciales o solo horas iguales, si convierte las horas a una escala comparable en lugar de compararlas como texto, y si su resultado coincide con los 70 pares y 7 estudiantes de la referencia.

## Conclusiones

1. **Calidad de los datos.** La base es consistente en sus llaves, pero tiene problemas que afectan el análisis: horas guardadas como texto en formato de 12 horas (con una inconsistencia, `04:00PM`), dos cursos sin facultad, errores de digitación en nombres de cursos y un ordenamiento sensible a tildes y mayúsculas. Además, SQLite no hace cumplir las llaves foráneas por defecto.
2. **Oferta frente a demanda.** La oferta está sobredimensionada: 35 cursos del catálogo no tienen horario, 13 cursos programados no tienen estudiantes (23 sesiones sin uso) y la ocupación global de los cupos es de 8,7 %, sin ninguna sesión por encima del 30 %.
3. **Estudiantes.** Dos estudiantes de primer semestre no tienen matrícula y siete de los ocho matriculados tienen cruces de horario (70 pares en total), lo que indica que el proceso de inscripción no valida horarios.
4. **Docentes.** Los 10 profesores tienen asignación, pero la carga es desigual: Saanvi Bahl orienta 8 cursos y 18 sesiones semanales, frente a 1 curso de Charlotte Doppler.
5. **SQL y agentes de IA.** Las consultas que combinan dos tablas de relación muchos a muchos deben contar por separado (con CTE o subconsultas) para evitar el producto cartesiano. Las preguntas en lenguaje natural sobre horarios esconden detalles (formato de las horas, solapamientos parciales, autocomparaciones) que un agente Text-to-SQL puede pasar por alto, así que toda consulta generada por IA debe validarse contra una consulta de referencia antes de usarse para tomar decisiones.

## Exportación a HTML

Guardar el notebook antes de ejecutar la celda. El archivo HTML queda en la misma carpeta del notebook.

In [55]:
import subprocess
import sys

conn.close()

ruta_nb = ('/content/drive/MyDrive/Colab Notebooks/Taller_3_SQL_Analitica_Academica.ipynb'
           if EN_COLAB else 'Taller_3_SQL_Analitica_Academica.ipynb')
salida = subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'html',
                         '--HTMLExporter.exclude_anchor_links=True', ruta_nb],
                        capture_output=True, text=True)
print(salida.stderr)

[NbConvertApp] Converting notebook Taller_3_SQL_Analitica_Academica.ipynb to html
[NbConvertApp] Writing 437269 bytes to Taller_3_SQL_Analitica_Academica.html

